# Diversity check for geometric data filtering

Kaggle-ready notebook: checks whether selection by geometric metrics reduces linguistic/task diversity.

It follows the idea from the quoted paper: extract a root verb and first direct/object-like noun from each instruction, then compare distributions for different selection strategies.


In [1]:

# Kaggle setup
!pip -q install datasets sentence-transformers transformers accelerate faiss-cpu plotly kaleido spacy tqdm scipy scikit-learn

# spaCy model for verb/noun extraction. If internet is disabled, the notebook will use a fallback heuristic.
!python -m spacy download en_core_web_sm -q || true


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.3 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:

import os, re, math, random, warnings, json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from scipy.stats import entropy

import plotly.express as px
import plotly.io as pio

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT_DIR = Path('/kaggle/working/geom_diversity_check')
PLOTS_DIR = OUT_DIR / 'plots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('output:', OUT_DIR)


device: cuda
output: /kaggle/working/geom_diversity_check


In [3]:

# =========================
# Config
# =========================

# Alpaca is convenient here because instructions usually have a clear action verb and object noun.
# For your own corpus, replace load_texts() below with a list of pretraining samples.
DATASET_NAME = 'yahma/alpaca-cleaned'
TEXT_FIELD = 'instruction'

N_TOTAL = 12000        # total examples used to compute embeddings/metrics; increase if Kaggle runtime allows
SUBSET_SIZE = 3000     # selection size, same as in the screenshot
KNN_K = 30             # local neighborhood size for geometric metrics
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# Quality baseline. False is faster. True computes GPT-2/distilgpt2 perplexity and selects easiest/high-quality-looking samples.
COMPUTE_PPL_QUALITY = True
PPL_MODEL = 'distilgpt2'
PPL_BATCH_SIZE = 16
PPL_MAX_LENGTH = 128

# Main geometric selectors to test.
# high_erank / high_id often picks locally complex or outlier-ish areas;
# low_erank / low_id often picks dense/redundant areas.
GEOM_SELECTORS = ['high_erank', 'low_erank', 'high_id', 'low_id']


In [4]:

def load_texts(dataset_name=DATASET_NAME, text_field=TEXT_FIELD, n_total=N_TOTAL):
    ds = load_dataset(dataset_name, split='train')
    if text_field not in ds.column_names:
        raise ValueError(f'{text_field=} not in dataset columns: {ds.column_names}')
    texts = []
    for x in ds[text_field]:
        if isinstance(x, str):
            x = re.sub(r'\s+', ' ', x).strip()
            if len(x) >= 8:
                texts.append(x)
    rng = np.random.default_rng(SEED)
    if len(texts) > n_total:
        idx = rng.choice(len(texts), size=n_total, replace=False)
        texts = [texts[i] for i in idx]
    return texts

texts = load_texts()
print('texts:', len(texts))
print(pd.Series(texts[:5], name='sample'))


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

texts: 12000
0    Can you name some popular shows airing this se...
1    Categorize each of the items below into either...
2    Suggest two activities that can be done outsid...
3    Change the verbs in brackets to a more appropr...
4    What is the source of the data for a typical m...
Name: sample, dtype: object


In [5]:

# =========================
# Embeddings
# =========================

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
emb = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
emb = emb.astype('float32')
print('embeddings:', emb.shape)
np.save(OUT_DIR / 'embeddings.npy', emb)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

embeddings: (12000, 384)


In [6]:

# =========================
# Local geometric metrics
# =========================

# We use two local geometry proxies:
# 1) local_id_twonn: TwoNN-like intrinsic dimensionality estimate from nearest-neighbor distance ratios.
# 2) local_erank: effective rank of covariance spectrum in the local kNN neighborhood.

n = len(texts)
k = min(KNN_K + 1, n)
nn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='auto')
nn.fit(emb)
dists, neigh = nn.kneighbors(emb, return_distance=True)

# remove self neighbor
knn_dists = dists[:, 1:]
knn_idx = neigh[:, 1:]

# local TwoNN-style ID: ID ~= 1 / mean(log(r2/r1)); here stabilized and computed from first several pairs
r1 = np.maximum(knn_dists[:, 0], 1e-8)
ratios = np.maximum(knn_dists[:, 1:10], 1e-8) / r1[:, None]
local_id = 1.0 / np.maximum(np.mean(np.log(np.maximum(ratios, 1.000001)), axis=1), 1e-6)
local_id = np.clip(local_id, 0, np.nanpercentile(local_id, 99.5))


def effective_rank_from_singular_values(s):
    s = np.maximum(s, 1e-12)
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p)).sum()))

local_erank = np.zeros(n, dtype=np.float32)
for i in tqdm(range(n), desc='local effective rank'):
    X = emb[knn_idx[i]]
    X = X - X.mean(axis=0, keepdims=True)
    # SVD on k x d local cloud; cheap because k is small
    s = np.linalg.svd(X, compute_uv=False)
    local_erank[i] = effective_rank_from_singular_values(s)

geom_df = pd.DataFrame({
    'text': texts,
    'local_id_twonn': local_id,
    'local_erank': local_erank,
    'mean_knn_cosine_distance': knn_dists.mean(axis=1),
})
geom_df.to_csv(OUT_DIR / 'geometry_metrics.csv', index=False)
geom_df.describe()


local effective rank:   0%|          | 0/12000 [00:00<?, ?it/s]

,local_id_twonn,local_erank,mean_knn_cosine_distance
count,12000.000000,12000.000000,12000.000000
mean,4.636373,26.021914,0.464601
std,3.805249,0.649760,0.093783
min,0.200678,22.793350,0.158037
25%,2.116419,25.624644,0.400862
50%,3.521866,26.089393,0.467608
75%,5.889376,26.482343,0.531368
max,24.465258,27.791590,0.774421


In [7]:

# =========================
# Optional quality baseline: perplexity
# =========================

# Lower perplexity is a common rough quality/fluency heuristic.
# This is not the core of the experiment, but it reproduces the paper's comparison idea:
# random vs quality-driven vs geometry-driven selection.

if COMPUTE_PPL_QUALITY:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tok = AutoTokenizer.from_pretrained(PPL_MODEL)
    model = AutoModelForCausalLM.from_pretrained(PPL_MODEL).to(DEVICE)
    model.eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    ppls = []
    for start in tqdm(range(0, len(texts), PPL_BATCH_SIZE), desc='perplexity'):
        batch = texts[start:start + PPL_BATCH_SIZE]
        enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=PPL_MAX_LENGTH).to(DEVICE)
        with torch.no_grad():
            out = model(**enc, labels=enc['input_ids'])
            # token-level loss per sample
            logits = out.logits[:, :-1, :].contiguous()
            labels = enc['input_ids'][:, 1:].contiguous()
            mask = enc['attention_mask'][:, 1:].contiguous()
            loss_tok = torch.nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                reduction='none'
            ).view(labels.size())
            loss = (loss_tok * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
            ppls.extend(torch.exp(loss).detach().cpu().numpy().tolist())
    geom_df['ppl'] = ppls
else:
    geom_df['ppl'] = np.nan

geom_df.to_csv(OUT_DIR / 'geometry_metrics_with_quality.csv', index=False)
geom_df[['local_id_twonn', 'local_erank', 'mean_knn_cosine_distance', 'ppl']].describe()


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

perplexity:   0%|          | 0/750 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


,local_id_twonn,local_erank,mean_knn_cosine_distance,ppl
count,12000.000000,12000.000000,12000.000000,12000.000000
mean,4.636373,26.021914,0.464601,209.770929
std,3.805249,0.649760,0.093783,519.790845
min,0.200678,22.793350,0.158037,7.810586
25%,2.116419,25.624644,0.400862,64.232943
50%,3.521866,26.089393,0.467608,111.133480
75%,5.889376,26.482343,0.531368,204.563889
max,24.465258,27.791590,0.774421,19405.412109


In [8]:

# =========================
# Verb-noun extraction
# =========================

try:
    import spacy
    nlp = spacy.load('en_core_web_sm', disable=['ner'])
    HAS_SPACY = True
    print('using spaCy parser')
except Exception as e:
    HAS_SPACY = False
    print('spaCy unavailable, using fallback heuristic:', repr(e))

STOP_NOUNS = set('thing things something anything everything way type kind example examples list information info text sentence word words paragraph'.split())


def fallback_verb_noun(text):
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", text.lower())
    if not words:
        return ('unknown', 'unknown')
    verb = words[0]
    noun = 'unknown'
    for w in words[1:]:
        if len(w) > 2 and w not in STOP_NOUNS:
            noun = w
            break
    return (verb, noun)


def extract_pair_doc(doc):
    # root verb of the main sentence
    root = None
    for sent in doc.sents:
        for tok in sent:
            if tok.dep_ == 'ROOT' and tok.pos_ in {'VERB', 'AUX'}:
                root = tok
                break
        if root is not None:
            break
    if root is None:
        for tok in doc:
            if tok.pos_ == 'VERB':
                root = tok
                break
    verb = root.lemma_.lower() if root is not None else 'unknown'

    noun = None
    if root is not None:
        # direct object first, then close object-like noun
        for tok in root.children:
            if tok.dep_ in {'dobj', 'obj', 'attr', 'oprd'} and tok.pos_ in {'NOUN', 'PROPN', 'PRON'}:
                noun = tok
                break
        if noun is None:
            for tok in doc:
                if tok.i > root.i and tok.pos_ in {'NOUN', 'PROPN'} and tok.lemma_.lower() not in STOP_NOUNS:
                    noun = tok
                    break
    if noun is None:
        for tok in doc:
            if tok.pos_ in {'NOUN', 'PROPN'} and tok.lemma_.lower() not in STOP_NOUNS:
                noun = tok
                break
    noun_lemma = noun.lemma_.lower() if noun is not None else 'unknown'
    return (verb, noun_lemma)

if HAS_SPACY:
    pairs = []
    for doc in tqdm(nlp.pipe(texts, batch_size=256), total=len(texts), desc='parse verb-noun'):
        pairs.append(extract_pair_doc(doc))
else:
    pairs = [fallback_verb_noun(t) for t in texts]

geom_df['verb'] = [p[0] for p in pairs]
geom_df['noun'] = [p[1] for p in pairs]
geom_df['verb_noun'] = [f'{v}::{u}' for v, u in pairs]
geom_df.to_csv(OUT_DIR / 'samples_with_pairs_and_metrics.csv', index=False)
geom_df[['text', 'verb', 'noun', 'verb_noun']].head(10)


using spaCy parser


parse verb-noun:   0%|          | 0/12000 [00:00<?, ?it/s]

,text,verb,noun,verb_noun
0,Can you name some popular shows airing this se...,name,show,name::show
1,Categorize each of the items below into either...,categorize,each,categorize::each
2,Suggest two activities that can be done outsid...,suggest,activity,suggest::activity
3,Change the verbs in brackets to a more appropr...,change,verb,change::verb
4,What is the source of the data for a typical m...,be,what,be::what
5,List five communication strategies for virtual...,unknown,communication,unknown::communication
6,Give two strategies for revising an essay.,give,strategy,give::strategy
7,List the members of the United Nations,list,member,list::member
8,Compose a funny headline for a newspaper article.,compose,headline,compose::headline
9,Compose a sentence with a specific tone.,compose,sentence,compose::sentence


In [9]:

# =========================
# Selection strategies
# =========================

def select_top(col, size=SUBSET_SIZE, largest=True):
    vals = geom_df[col].to_numpy()
    order = np.argsort(vals)
    if largest:
        order = order[::-1]
    return order[:size]

rng = np.random.default_rng(SEED)
all_idx = np.arange(len(geom_df))

selections = {
    'random': rng.choice(all_idx, size=SUBSET_SIZE, replace=False),
    'geom_high_erank': select_top('local_erank', largest=True),
    'geom_low_erank': select_top('local_erank', largest=False),
    'geom_high_id': select_top('local_id_twonn', largest=True),
    'geom_low_id': select_top('local_id_twonn', largest=False),
    'geom_outlier_high_knn_dist': select_top('mean_knn_cosine_distance', largest=True),
    'geom_dense_low_knn_dist': select_top('mean_knn_cosine_distance', largest=False),
}

if COMPUTE_PPL_QUALITY:
    selections['quality_low_ppl'] = select_top('ppl', largest=False)
    selections['quality_high_ppl'] = select_top('ppl', largest=True)

# Save selected ids
sel_rows = []
for name, idx in selections.items():
    for i in idx:
        sel_rows.append({'selector': name, 'idx': int(i)})
pd.DataFrame(sel_rows).to_csv(OUT_DIR / 'selected_indices.csv', index=False)

{key: len(val) for key, val in selections.items()}


{'random': 3000,
 'geom_high_erank': 3000,
 'geom_low_erank': 3000,
 'geom_high_id': 3000,
 'geom_low_id': 3000,
 'geom_outlier_high_knn_dist': 3000,
 'geom_dense_low_knn_dist': 3000,
 'quality_low_ppl': 3000,
 'quality_high_ppl': 3000}

In [10]:

# =========================
# Diversity metrics
# =========================


def gini_from_counts(counts):
    x = np.array(list(counts), dtype=np.float64)
    if len(x) == 0 or x.sum() == 0:
        return np.nan
    x = np.sort(x)
    n = len(x)
    return (2 * np.arange(1, n + 1) @ x) / (n * x.sum()) - (n + 1) / n


def diversity_report(idx):
    sub = geom_df.iloc[idx]
    pair_counts = Counter(sub['verb_noun'])
    verb_counts = Counter(sub['verb'])
    noun_counts = Counter(sub['noun'])
    pair_freq = np.array(list(pair_counts.values()), dtype=float)
    pair_prob = pair_freq / pair_freq.sum()
    return {
        'n': len(sub),
        'unique_pairs': len(pair_counts),
        'unique_verbs': len(verb_counts),
        'unique_nouns': len(noun_counts),
        'pair_entropy': entropy(pair_prob, base=2),
        'pair_perplexity': 2 ** entropy(pair_prob, base=2),
        'top_pair_share': pair_freq.max() / pair_freq.sum(),
        'top10_pair_share': pair_freq[np.argsort(pair_freq)[-10:]].sum() / pair_freq.sum(),
        'pair_gini': gini_from_counts(pair_freq),
        'mean_local_erank': sub['local_erank'].mean(),
        'mean_local_id': sub['local_id_twonn'].mean(),
        'mean_knn_dist': sub['mean_knn_cosine_distance'].mean(),
        'mean_ppl': sub['ppl'].mean() if 'ppl' in sub else np.nan,
    }

summary = pd.DataFrame([
    {'selector': name, **diversity_report(idx)}
    for name, idx in selections.items()
]).sort_values('unique_pairs', ascending=False)

summary.to_csv(OUT_DIR / 'diversity_summary.csv', index=False)
summary


,selector,n,unique_pairs,unique_verbs,unique_nouns,pair_entropy,pair_perplexity,top_pair_share,top10_pair_share,pair_gini,mean_local_erank,mean_local_id,mean_knn_dist,mean_ppl
5,geom_outlier_high_knn_dist,3000,1999,302,1053,10.136055,1125.269315,0.075667,0.142000,0.313592,26.362032,6.142437,0.581885,220.225978
8,quality_high_ppl,3000,1983,288,883,10.448387,1397.262282,0.020667,0.079000,0.306637,26.002525,4.589523,0.459796,557.363867
3,geom_high_id,3000,1962,307,856,10.238254,1207.874129,0.041333,0.125667,0.321226,26.102703,9.766234,0.503437,207.626929
0,random,3000,1865,263,771,10.180496,1160.472402,0.037667,0.117333,0.344092,26.028601,4.718212,0.463977,205.841856
1,geom_high_erank,3000,1849,240,862,10.081882,1083.799133,0.053000,0.129667,0.352392,26.782984,5.061535,0.517985,207.868525
4,geom_low_id,3000,1657,230,691,9.968559,1001.924851,0.030667,0.118000,0.393702,25.950209,1.439938,0.436825,233.551734
2,geom_low_erank,3000,1620,235,617,9.764344,869.681719,0.044667,0.151333,0.413688,25.148380,4.189174,0.413532,199.045450
7,quality_low_ppl,3000,1575,215,666,9.459180,703.877407,0.082333,0.199667,0.437242,26.064888,4.690032,0.476436,44.300581
6,geom_dense_low_knn_dist,3000,1391,203,390,9.517923,733.128769,0.040333,0.141333,0.466659,25.662029,3.389240,0.342142,195.429539


In [11]:

# =========================
# Plots: bar metrics
# =========================

plot_cols = ['unique_pairs', 'unique_verbs', 'unique_nouns', 'pair_entropy', 'pair_perplexity', 'top10_pair_share', 'pair_gini']
for col in plot_cols:
    fig = px.bar(
        summary.sort_values(col, ascending=False),
        x='selector', y=col,
        title=f'Diversity metric: {col}',
        text=col,
    )
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig.update_layout(xaxis_tickangle=-35, height=520)
    fig.show()
    fig.write_html(PLOTS_DIR / f'{col}.html')
    try:
        fig.write_image(PLOTS_DIR / f'{col}.png', scale=2)
    except Exception as e:
        print('png save failed:', col, e)


png save failed: unique_pairs 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: unique_verbs 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: unique_nouns 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: pair_entropy 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: pair_perplexity 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: top10_pair_share 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: pair_gini 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [12]:

# =========================
# Sunburst distributions like in the paper
# =========================

# For readability, rare pairs are grouped into __other__.
def prepare_sunburst_df(idx, top_verbs=18, top_pairs_per_verb=8):
    sub = geom_df.iloc[idx].copy()
    top_v = set(sub['verb'].value_counts().head(top_verbs).index)
    sub['verb_plot'] = sub['verb'].where(sub['verb'].isin(top_v), '__other_verbs__')

    rows = []
    for verb, g in sub.groupby('verb_plot'):
        top_pairs = set(g['verb_noun'].value_counts().head(top_pairs_per_verb).index)
        gg = g.copy()
        gg['noun_plot'] = [n if p in top_pairs else '__other_nouns__' for n, p in zip(gg['noun'], gg['verb_noun'])]
        rows.append(gg)
    out = pd.concat(rows)
    return out.groupby(['verb_plot', 'noun_plot'], as_index=False).size().rename(columns={'size': 'count'})

sunburst_names = ['random', 'quality_low_ppl' if 'quality_low_ppl' in selections else None, 'geom_high_erank', 'geom_low_erank', 'geom_high_id', 'geom_low_id']
sunburst_names = [x for x in sunburst_names if x is not None]

for name in sunburst_names:
    dfp = prepare_sunburst_df(selections[name])
    fig = px.sunburst(
        dfp,
        path=['verb_plot', 'noun_plot'],
        values='count',
        title=f'Verb-noun diversity: {name}'
    )
    fig.update_layout(height=680)
    fig.show()
    fig.write_html(PLOTS_DIR / f'sunburst_{name}.html')
    try:
        fig.write_image(PLOTS_DIR / f'sunburst_{name}.png', scale=2)
    except Exception as e:
        print('png save failed:', name, e)


png save failed: random 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: quality_low_ppl 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: geom_high_erank 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: geom_low_erank 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: geom_high_id 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



png save failed: geom_low_id 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [13]:

# =========================
# Top pairs table
# =========================

top_tables = []
for name, idx in selections.items():
    vc = geom_df.iloc[idx]['verb_noun'].value_counts().head(20)
    tmp = vc.reset_index()
    tmp.columns = ['verb_noun', 'count']
    tmp['selector'] = name
    tmp['share'] = tmp['count'] / SUBSET_SIZE
    top_tables.append(tmp)

top_pairs_df = pd.concat(top_tables, ignore_index=True)
top_pairs_df.to_csv(OUT_DIR / 'top_verb_noun_pairs_by_selector.csv', index=False)

top_pairs_df.head(60)


,verb_noun,count,selector,share
0,be::what,113,random,0.037667
1,generate::list,53,random,0.017667
2,rewrite::sentence,38,random,0.012667
3,give::example,26,random,0.008667
4,generate::sentence,23,random,0.007667
5,write::story,22,random,0.007333
6,provide::example,22,random,0.007333
7,create::list,21,random,0.007000
8,edit::sentence,19,random,0.006333
9,explain::concept,15,random,0.005000


In [14]:

# =========================
# Automatic conclusion
# =========================

random_row = summary.set_index('selector').loc['random']
conclusion_rows = []
for _, row in summary.iterrows():
    if row['selector'] == 'random':
        continue
    conclusion_rows.append({
        'selector': row['selector'],
        'unique_pairs_vs_random_%': 100 * (row['unique_pairs'] / random_row['unique_pairs'] - 1),
        'entropy_vs_random_%': 100 * (row['pair_entropy'] / random_row['pair_entropy'] - 1),
        'top10_share_vs_random_%': 100 * (row['top10_pair_share'] / random_row['top10_pair_share'] - 1),
        'gini_vs_random_%': 100 * (row['pair_gini'] / random_row['pair_gini'] - 1),
    })

conclusion = pd.DataFrame(conclusion_rows).sort_values('unique_pairs_vs_random_%')
conclusion.to_csv(OUT_DIR / 'diversity_vs_random.csv', index=False)
print(conclusion.to_string(index=False))

worst = conclusion.iloc[0]
print('\nMain read:')
print(f"Worst diversity by unique verb-noun pairs: {worst['selector']}")
print(f"Unique pairs vs random: {worst['unique_pairs_vs_random_%']:.1f}%")
print(f"Entropy vs random: {worst['entropy_vs_random_%']:.1f}%")
print(f"Top-10 concentration vs random: {worst['top10_share_vs_random_%']:.1f}%")

if worst['unique_pairs_vs_random_%'] < 0 and worst['entropy_vs_random_%'] < 0:
    print('Conclusion: this selector reduces diversity relative to random selection.')
else:
    print('Conclusion: no clear diversity collapse for the worst selector under this metric; inspect plots and try another geometry score/budget.')


                  selector  unique_pairs_vs_random_%  entropy_vs_random_%  top10_share_vs_random_%  gini_vs_random_%
   geom_dense_low_knn_dist                -25.415550            -6.508265                20.454545         35.620328
           quality_low_ppl                -15.549598            -7.085275                70.170455         27.071270
            geom_low_erank                -13.136729            -4.087746                28.977273         20.225932
               geom_low_id                -11.152815            -2.081803                 0.568182         14.417662
           geom_high_erank                 -0.857909            -0.968664                10.511364          2.412268
              geom_high_id                  5.201072             0.567339                 7.102273         -6.645292
          quality_high_ppl                  6.327078             2.631410               -32.670455        -10.885112
geom_outlier_high_knn_dist                  7.184987            

In [15]:

# Zip all outputs for download from Kaggle sidebar
!cd /kaggle/working && zip -qr geom_diversity_check.zip geom_diversity_check
print('/kaggle/working/geom_diversity_check.zip')


/kaggle/working/geom_diversity_check.zip
